# Practical 04a — Uninformed Search: BFS and DFS on Puzzles and Mazes (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 4 |
| **Related lectures** | Lectures 9, 10 and 12 |
| **Duration** | 3 hours |
| **Covers the same work as** | Practical 04 (same concepts, same five tasks, same marks) |
| **Submission** | This notebook, run top to bottom from a clean kernel |

This is an easier route through Practical 04. Nothing is left out — you still describe a
search problem, write one search loop that does both breadth-first and depth-first search,
build your own maze, trick depth-first search, and break the search on purpose — but every
idea comes in a small step, most of the code is written for you, and you fill in the
missing pieces. A **CHECK** cell after each exercise tells you whether you got it right.

## 0. How to use this notebook

1. **Run cells in order**, top to bottom. Click a cell and press **Shift + Enter**.
2. When you see **`___`** (three underscores) in code, that is a **blank** for you to fill in.
   Replace the `___` with your answer, then run the cell again.
3. Run the **CHECK** cell that follows. It prints:
   - `OK` — correct, move on.
   - `NOT YET` — it shows what it expected, what you gave, and a hint.
4. If you see a red error message, the most common cause is a blank you have not filled yet.
   Read the **last line** of the error — it names the problem.
5. If you restart the notebook (Kernel → Restart), run all the cells above the one you are
   on again, starting from the setup cell.

> 💡 **Predict before you run.** Many cells ask *"What do you think this will print?"*
> Say your guess out loud (or write it in a comment) before pressing Shift + Enter. This is
> the fastest way to learn.

## 1. Objectives

By the end of this session you will be able to:

1. Define a search problem by its four components.
2. Design a state representation and justify what you left out of it.
3. Implement the generic search algorithm with an interchangeable frontier.
4. Produce breadth-first and depth-first search by changing only the frontier.
5. Measure and compare nodes expanded and solution quality.

## 2. Background — the idea in plain words

**A computer can find its way through a maze without "seeing" it — by trying moves in an
organised order and remembering which routes it has not finished yet.**

Here is the maze you will solve. `S` is the start, `G` the goal, `#` a wall, `.` an open cell.

```
#########
#S..#...#
#.#.#.#.#
#.#...#.#
#.#####.#
#.......#
#.#####G#
#########
```

**A search problem has four parts (Lecture 9):**

| Part | Question it answers | For this maze |
|---|---|---|
| **Start state** | Where do we begin? | the cell `S` |
| **Actions** | From here, where can I go? | one step up, down, left or right — if it is not a wall |
| **Goal test** | Am I there yet? | is this cell `G`? |
| **Cost** | What does a move cost? | 1 per step |

**Words used in this lab:**

| Word | What it means here | Example |
|---|---|---|
| **state** | a description of *where we are now* — nothing more | `(1, 1)` = row 1, column 1 |
| **path** | the list of states visited so far, start first | `[(1, 1), (2, 1), (3, 1)]` |
| **frontier** | the paths we have found but **not yet followed further** | a list of paths |
| **expand** | take a path off the frontier and add one longer path for each move from its end | — |
| **nodes expanded** | how many times we expanded — our measure of **work** | `24` |
| **visited** | the set of states already expanded, so we never expand one twice | `{(1, 1), (2, 1)}` |

**Breadth-first vs depth-first — one tiny difference (Lecture 10):**

| | Breadth-first search (BFS) | Depth-first search (DFS) |
|---|---|---|
| Takes from the frontier | the **oldest** path | the **newest** path |
| Like | a **queue** at the bank — first come, first served | a **stack of plates** — the last one put on is taken first |
| Python | `frontier.pop(0)` | `frontier.pop()` |
| Explores | all routes of length 1, then 2, then 3 … | one route as deep as it goes, then backs up |
| Finds the shortest route? | **yes** (when every step costs 1) | not necessarily |

Everything else in the two searches is **identical**. That is the main lesson of today.

**Key points to keep in mind while you work:**

- The state must contain everything affecting the future and nothing else.
- Do not put the path travelled inside the state, or repeated positions can never be recognised.
- Measure nodes expanded, not seconds: node counts are comparable across machines.
- Without a repeated-state check, depth-first search can loop forever on a maze with cycles.

## 3. Setup

**Software:** Python 3.10 or later. No packages to install.

Run the cell below **first, every time you open the notebook.** It creates the blank
`___` and a small `check()` helper that the CHECK cells use. You do not need to understand
how `check()` works.

In [1]:
___ = None          # the blank you will replace with your answers

def check(label, got, expected, hint=''):
    """Print OK if got == expected, otherwise NOT YET with a hint."""
    if got == expected:
        print('OK      ', label)
    else:
        print('NOT YET ', label)
        print('          expected:', repr(expected))
        print('          you gave:', repr(got))
        if hint:
            print('          hint    :', hint)

print('Setup done. You are ready to start.')

Setup done. You are ready to start.


## 4. Python warm-up — only what you need today

Six small Python ideas are used in this lab. Each has a demo cell (just run it) and a tiny
exercise. If you already know them, run the cells quickly and move on.

### Warm-up 1 — A position is a tuple `(row, col)`

We describe a cell by its **row** and **column**, counted from 0 at the top-left. Rows grow
**downwards**, columns grow **to the right**. A move is a small change `(dr, dc)` added to
the position:

| Move | `(dr, dc)` | From `(3, 5)` you reach |
|---|---|---|
| up | `(-1, 0)` | `(2, 5)` |
| down | `(1, 0)` | `(4, 5)` |
| left | `(0, -1)` | `(3, 4)` |
| right | `(0, 1)` | `(3, 6)` |

In [ ]:
pos = (3, 5)              # row 3, column 5
r, c = pos                # unpacking: r gets 3, c gets 5
print('row', r, 'col', c)

dr, dc = (0, 1)           # one step RIGHT: same row, column + 1
print('one step right:', (r + dr, c + dc))

**Your turn:** from `(3, 5)`, write the cell one step **up** and the cell one step **left**.

In [ ]:
above   = (2, 5)
left_of = (3, 4)

In [ ]:
# CHECK warm-up 1
check('one step up', above, (2, 5), 'up means the row goes DOWN by 1: (3 - 1, 5)')
check('one step left', left_of, (3, 4), 'left means the column goes down by 1: (3, 5 - 1)')

### Warm-up 2 — A maze is a list of strings

Each string is one row. `MAZE[r]` is row `r`, and `MAZE[r][c]` is the single character at
row `r`, column `c`. `enumerate` gives you the number **and** the item as you loop.

In [ ]:
TINY = ['#####',
        '#S.G#',
        '#####']

print(TINY[1])            # row 1 -> '#S.G#'
print(TINY[1][1])         # row 1, column 1 -> 'S'

for r, row in enumerate(TINY):
    for c, ch in enumerate(row):
        if ch == 'G':
            print('G is at', (r, c))

**Your turn — predict:** what character is at `TINY[1][2]`, and how many `#` walls does `TINY` have in total?

In [ ]:
char_1_2  = '.'
num_walls = 12

In [ ]:
# CHECK warm-up 2
check('TINY[1][2]', char_1_2, TINY[1][2], "row 1 is '#S.G#'; count from 0: #=0, S=1, .=2")
check('number of walls', num_walls, sum(row.count('#') for row in TINY),
      'row 0 has 5, row 1 has 2 (the two ends), row 2 has 5')

### Warm-up 3 — Queue or stack: `pop(0)` versus `pop()`

This is **the** most important warm-up. A list can give items back in two orders:

- `items.pop(0)` removes the **first** item — the **oldest**. That is a **queue** (BFS).
- `items.pop()` removes the **last** item — the **newest**. That is a **stack** (DFS).

In [ ]:
line = ['A', 'B', 'C']        # A arrived first, C arrived last
print(line.pop(0))            # A  - oldest first, like a queue at the bank
print(line)                   # ['B', 'C']

plates = ['A', 'B', 'C']      # C was put on the pile last
print(plates.pop())           # C  - newest first, like a stack of plates
print(plates)                 # ['A', 'B']

**Your turn — predict:** for `items = ['P', 'Q', 'R', 'S']`, what does each line give back?

In [ ]:
items = ['P', 'Q', 'R', 'S']
first_from_queue = 'P'       # items.pop(0)
first_from_stack = 'S'       # items.pop()

items.append('T')              # a new item arrives...
after_append_stack = 'T'     # ...what does items.pop() give now?

In [ ]:
# CHECK warm-up 3
check('pop(0)', first_from_queue, ['P', 'Q', 'R', 'S'].pop(0), 'pop(0) takes the FIRST (oldest) item')
check('pop()', first_from_stack, ['P', 'Q', 'R', 'S'].pop(), 'pop() takes the LAST (newest) item')
check('pop() after append', after_append_stack, ['P', 'Q', 'R', 'S', 'T'].pop(),
      'append puts T at the END, and pop() takes from the end')

### Warm-up 4 — Sets remember what we have seen

A **set** has no repeats. `seen.add(x)` puts `x` in; `x in seen` asks whether it is there.
Tuples such as `(1, 1)` can go in a set, so a set of positions is a perfect "already
visited" list.

In [ ]:
visited = set()
visited.add((1, 1))
visited.add((1, 1))           # adding it again changes nothing
print(visited)
print((1, 1) in visited)      # True
print((2, 1) in visited)      # False

**Your turn:** add `(2, 1)`, then predict the answers.

In [ ]:
visited.add((2, 1))
now_seen   = True        # is (2, 1) in visited now?
how_many   = 2           # len(visited)

In [ ]:
# CHECK warm-up 4
check('(2, 1) added', (2, 1) in visited, True, 'visited.add((2, 1)) - note the double brackets')
check('now_seen', now_seen, (2, 1) in visited, 'you just added it')
check('how_many', how_many, len(visited), '(1, 1) once (repeats do not count) and (2, 1)')

### Warm-up 5 — A path is a list; `path[-1]` is where you are now

The search keeps **paths**, not just cells. `path[-1]` is the **last** item: the cell at the
end of the path. `path + [nxt]` makes a **new**, longer list — the old path is left as it
was, which is exactly what we want, because other routes may still need it.

In [ ]:
path = [(1, 1), (2, 1)]
print('now at:', path[-1])

longer = path + [(3, 1)]
print('old path:', path)          # unchanged
print('new path:', longer)
print('steps taken on the new path:', len(longer) - 1)

**Your turn — predict:**

In [ ]:
# longer = [(1, 1), (2, 1), (3, 1)]
end_of_longer = (3, 1)     # longer[-1]
length_of_old = 2          # len(path) - did path change?

In [ ]:
# CHECK warm-up 5
check('longer[-1]', end_of_longer, longer[-1], 'the last item of the list')
check('len(path)', length_of_old, len(path), 'path + [x] makes a NEW list; path itself keeps 2 items')

### Warm-up 6 — `yield`: hand back answers one at a time

A function with `yield` hands back several values, one after another. `list(...)` collects
them all. Our maze uses this to hand back every legal move from a cell.

In [ ]:
def evens_up_to(n):
    for k in range(0, n + 1, 2):
        yield k                    # hand back k, then carry on

print(list(evens_up_to(8)))        # [0, 2, 4, 6, 8]

**Your turn:** make `squares(n)` hand back `0, 1, 4, 9, …` — the square of each `k` from `0` to `n`.

In [ ]:
def squares(n):
    for k in range(n + 1):
        yield k * k

print(list(squares(4)))

In [ ]:
# CHECK warm-up 6
check('squares(4)', list(squares(4)), [0, 1, 4, 9, 16], 'yield k * k')

## 5. Procedure — build the search one piece at a time

Each step below introduces **one** idea. Run the demo, read the explanation, then do the
small exercise and its CHECK.

### Step 1 — Read the maze, and decide what a state is

`parse` walks over every character of the maze and collects three things: the set of
**wall** positions, the **start** position and the **goal** position. Fill in the **three**
blanks — each one is the current position.

**The state is `(row, col)` — nothing else.** The route taken so far is *not* part of the
state: the search keeps track of that separately (Lecture 9 explained why, and Task 5 shows
what goes wrong if you ignore it).

In [ ]:
MAZE = [
    '#########',
    '#S..#...#',
    '#.#.#.#.#',
    '#.#...#.#',
    '#.#####.#',
    '#.......#',
    '#.#####G#',
    '#########',
]

def parse(maze):
    walls, start, goal = set(), None, None
    for r, row in enumerate(maze):
        for c, ch in enumerate(row):
            if ch == '#':
                walls.add((r, c))          # blank 1: remember this wall
            elif ch == 'S':
                start = (r, c)             # blank 2: the start position
            elif ch == 'G':
                goal = (r, c)              # blank 3: the goal position
    return walls, start, goal

WALLS, START, GOAL = parse(MAZE)
print('start:', START, '  goal:', GOAL, '  walls:', len(WALLS))
print('a state looks like:', START)

**Exercise 1:** how many **open** cells (not walls) does the maze have? The maze is 9
columns wide and 8 rows tall.

In [ ]:
open_cells = 26

In [ ]:
# CHECK Step 1
check('START', START, (1, 1), 'blank 2 is (r, c)')
check('GOAL', GOAL, (6, 7), 'blank 3 is (r, c)')
check('number of walls', len(WALLS), 46, 'blank 1 is (r, c) - walls.add((r, c))')
check('open cells', open_cells, len(MAZE) * len(MAZE[0]) - len(WALLS),
      '9 x 8 = 72 cells in total, minus the walls')

### Step 2 — The four components, written as code

A search algorithm only ever asks the problem **four questions**. Here they are as the
methods of a class:

| Component | Method | Returns |
|---|---|---|
| Start state | `start_state()` | `(1, 1)` |
| Goal test | `is_goal(state)` | `True` or `False` |
| Actions | `actions(state)` | each legal `(next_state, cost)` pair, via `yield` |
| Cost | the `1` in each pair | 1 per step |

Any search written against these four methods works on **any** problem that provides them.
Fill in the **two** blanks.

In [ ]:
class MazeProblem:
    def __init__(self, walls, start, goal):
        self.walls, self.start, self.goal = walls, start, goal

    def start_state(self):
        return self.start

    def is_goal(self, state):
        return state == self.goal            # blank 1: are we at the goal?

    def actions(self, state):
        """Yield (next_state, cost) for each legal move."""
        r, c = state
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:     # up, down, left, right
            nxt = (r + dr, c + dc)
            if nxt not in self.walls:                        # blank 2: only if not a wall
                yield nxt, 1

problem = MazeProblem(WALLS, START, GOAL)
print('is the start the goal?', problem.is_goal(START))

**Exercise 2 — predict, then check.** What does `list(problem.actions(START))` give?
Look at the maze: from `S` at `(1, 1)`, which of up, down, left, right are open? Keep the
order **up, down, left, right**, and remember each item is `(next_state, cost)`.

In [ ]:
predicted_moves = [((2, 1), 1), ((1, 2), 1)]

print('actual:', list(problem.actions(START)))

In [ ]:
# CHECK Step 2
check('is_goal(GOAL)', problem.is_goal(GOAL), True, 'blank 1: return state == self.goal')
check('is_goal(START)', problem.is_goal(START), False, 'blank 1: return state == self.goal')
check('moves from START', predicted_moves, list(problem.actions(START)),
      'up (0,1) and left (1,0) are walls; down is (2,1), right is (1,2); each with cost 1')

### Step 3 — One search loop, two behaviours

**First, by hand.** The frontier starts with one path: `[[(1, 1)]]`. We take it off and
expand it: one new path for each move from `(1, 1)`. The frontier is now

```
[ [(1, 1), (2, 1)],        <- added first  (the "down" move)  - oldest
  [(1, 1), (1, 2)] ]       <- added last   (the "right" move) - newest
```

**Exercise 3a:** which path does each strategy take off the frontier **next**?

In [ ]:
frontier_now = [[(1, 1), (2, 1)], [(1, 1), (1, 2)]]

bfs_takes_next = [(1, 1), (2, 1)]      # frontier_now.pop(0)
dfs_takes_next = [(1, 1), (1, 2)]      # frontier_now.pop()

In [ ]:
# CHECK Step 3a
check('BFS takes', bfs_takes_next, [[(1, 1), (2, 1)], [(1, 1), (1, 2)]].pop(0),
      'BFS takes the OLDEST path: pop(0)')
check('DFS takes', dfs_takes_next, [[(1, 1), (2, 1)], [(1, 1), (1, 2)]].pop(),
      'DFS takes the NEWEST path: pop()')

**Now the code.** The recipe:

1. The frontier starts with one path holding just the start state.
2. Take a path off the frontier — the **oldest** for BFS, the **newest** for DFS.
3. Look at the state at the end of it. If we have **already expanded** that state, skip it.
   Otherwise remember it in `visited`, and count one more node **expanded**.
4. If it is the goal, return the path. Otherwise add one **longer path** for each move.
5. If the frontier runs empty, there is no route.

The line `if nxt not in path` stops a path walking back over **its own** cells (a *cycle*
check, Lecture 12). Fill in the **four** blanks.

In [ ]:
def search(problem, strategy='bfs', use_visited=True):
    """Generic search. strategy='bfs' takes the oldest path, 'dfs' the newest."""
    frontier = [[problem.start_state()]]      # a list of PATHS
    visited = set()
    expanded = 0
    while frontier:
        if strategy == 'bfs':
            path = frontier.pop(0)          # blank 1: oldest  -> breadth-first
        else:
            path = frontier.pop()           # blank 2: newest  -> depth-first
        state = path[-1]
        if use_visited:
            if state in visited:
                continue                      # already expanded: skip it
            visited.add(state)              # blank 3: remember this state
        expanded += 1
        if problem.is_goal(state):
            return path, expanded
        for nxt, cost in problem.actions(state):
            if nxt not in path:               # simple cycle check
                frontier.append(path + [nxt])   # blank 4: one step longer
    return None, expanded

for strat in ['bfs', 'dfs']:
    path, expanded = search(problem, strat)
    print('%-4s  path length %2d   nodes expanded %3d'
          % (strat.upper(), len(path), expanded))

In [ ]:
# CHECK Step 3
bfs_path, bfs_expanded = search(problem, 'bfs')
dfs_path, dfs_expanded = search(problem, 'dfs')
check('BFS path length', len(bfs_path), 12, 'blanks 1 and 4: pop(0), and append path + [nxt]')
check('BFS nodes expanded', bfs_expanded, 24, 'blank 3: visited.add(state)')
check('DFS path length', len(dfs_path), 16, 'blank 2: frontier.pop()')
check('DFS nodes expanded', dfs_expanded, 26, 'blank 2: frontier.pop()')

**Watch it run (given — just read and run).** This copy of the loop prints the first few
expansions, so you can see BFS spreading out evenly while DFS dives down one corridor.

In [ ]:
def trace(problem, strategy, steps=8):
    frontier = [[problem.start_state()]]
    visited = set()
    for i in range(steps):
        path = frontier.pop(0) if strategy == 'bfs' else frontier.pop()
        state = path[-1]
        if state in visited:
            continue
        visited.add(state)
        for nxt, cost in problem.actions(state):
            if nxt not in path:
                frontier.append(path + [nxt])
        print('%s expands %-8s  (path of %d steps)   frontier now holds %d paths'
              % (strategy.upper(), state, len(path) - 1, len(frontier)))
    print()

trace(problem, 'bfs')
trace(problem, 'dfs')

Look at the "path of … steps" column. BFS expands everything 1 step away, then 2, then 3 —
it moves forward in **layers**. DFS keeps going 1, 2, 3, 4, 5 … along one corridor before
it ever comes back.

### Step 4 — Seeing the difference

`show` draws a route on the maze with `*`. A **path length** of 12 means the path list holds
12 cells — the start, the goal, and 10 in between — so it takes **11 steps**.

In [ ]:
def show(maze, path, title):
    grid = [list(row) for row in maze]
    for (r, c) in path[1:-1]:
        grid[r][c] = '*'
    print(title)
    print('\n'.join(''.join(row) for row in grid))
    print()

for strat in ['bfs', 'dfs']:
    path, _ = search(problem, strat)
    show(MAZE, path, '%s route (length %d)' % (strat.upper(), len(path)))

**Exercise 4:** answer from what you see above.

In [ ]:
bfs_steps     = 11        # how many STEPS (moves) does the BFS route take?
shorter_route = 'BFS'     # which route is shorter: 'BFS' or 'DFS'?
dfs_did_more_work = True  # did DFS expand MORE nodes than BFS here? True or False

In [ ]:
# CHECK Step 4
check('BFS steps', bfs_steps, len(search(problem, 'bfs')[0]) - 1, 'path length 12 means 11 steps')
check('shorter route', shorter_route, 'BFS', 'BFS always finds the fewest steps when every step costs 1')
check('DFS did more work', dfs_did_more_work,
      search(problem, 'dfs')[1] > search(problem, 'bfs')[1], 'compare 26 with 24')

## 6. Tasks (marked)

These are the same five tasks as Practical 04, worth the same marks. Each one is split into
small parts, with blanks to fill in and a CHECK at the end.

### Task 1 — State your four components

> **Marks:** 3

**Part A:** fill in the four components of **this** maze problem. The CHECK compares your
answers with what the `problem` object actually does.

In [ ]:
t1_start_state    = (1, 1)     # the start state
t1_goal_state     = (6, 7)     # the one state for which is_goal(...) is True
t1_moves_from_start = 2        # how many legal moves are there from the start?
t1_cost_per_move  = 1          # what does one move cost?

In [ ]:
# CHECK Task 1, Part A
check('start state', t1_start_state, problem.start_state(), 'the S cell')
check('goal state', t1_goal_state, problem.goal, 'the G cell')
check('moves from start', t1_moves_from_start, len(list(problem.actions(START))), 'down and right')
check('cost per move', t1_cost_per_move, list(problem.actions(START))[0][1],
      'the second number in each (next_state, cost) pair')

**Part B — write in your own words**

| Component | For this maze |
|---|---|
| **Start state** | the cell `S`, i.e. `(1, 1)` |
| **Goal test** | the state equals the cell `G`, i.e. `(6, 7)` |
| **Actions** | step up, down, left or right into a neighbouring cell that is not a wall `#` |
| **Cost** | 1 for every step |

**Diagnostic question 1 — is anything *missing* from the state?**
The state is only `(row, col)`. To decide every legal move and whether we have reached the
goal, we need to know only the current position (the walls and goal are fixed parts of the problem, not of the state). Two different routes that arrive at the same cell face the
same future because the moves available from a cell and the goal test depend only on that cell, not on how we got there.

**Diagnostic question 2 — is anything *unnecessary* in the state?**
Both numbers are needed because together they identify one cell (a row alone or a column alone is ambiguous). The route travelled is **not** in the state,
because it does not change what can happen next; putting it in would make the same cell look like many different states, so repeated positions could never be recognised.

### Task 2 — Design your own maze

> **Marks:** 4

You need a maze **at least 15 × 15** with **several routes** to the goal.

To get you started, a 15 × 15 maze `STARTER` is given below. **Copy it into `MY_MAZE` and
change it** — open some walls (`#` → `.`) to make short cuts, or close some open cells
(`.` → `#`). Rules:

- change **at least 2 cells** (the CHECK counts them) — more is better;
- keep the outer border all `#`, and keep exactly one `S` and one `G`;
- every row must have the same length;
- there must be **at least 2 loops**, so that several routes to the goal exist.

> **Tip:** an easy way to make a second route is to open a wall between two corridors that
> run side by side.

In [ ]:
# A 15 x 15 starter maze (given), and MY_MAZE: a copy with 3 walls opened.
STARTER = [
    '###############',
    '#S#.......#...#',
    '#.#.#.###.#.#.#',
    '#.#.#.#.#.#.#.#',
    '#.#.#.#.#.###.#',
    '#.#.#.#.#.....#',
    '#.###.#.#####.#',
    '#.....#.....#.#',
    '#######.#.###.#',
    '#.....#.#.....#',
    '#.###.#.#######',
    '#.#...#.#...#.#',
    '#.#####.#.#.#.#',
    '#.........#..G#',
    '###############',
]

# MY_MAZE = STARTER with these walls opened (# -> .): [(6, 11), (8, 3), (11, 8)]
MY_MAZE = [
    '###############',
    '#S#.......#...#',
    '#.#.#.###.#.#.#',
    '#.#.#.#.#.#.#.#',
    '#.#.#.#.#.###.#',
    '#.#.#.#.#.....#',
    '#.###.#.###.#.#',
    '#.....#.....#.#',
    '###.###.#.###.#',
    '#.....#.#.....#',
    '#.###.#.#######',
    '#.#...#.....#.#',
    '#.#####.#.#.#.#',
    '#.........#..G#',
    '###############',
]

def count_loops(maze):
    """Number of independent loops = edges - open cells + 1 (maze is connected)."""
    open_cells = {(r, c) for r, row in enumerate(maze) for c, ch in enumerate(row) if ch != '#'}
    edges = sum(1 for (r, c) in open_cells for d in [(0, 1), (1, 0)]
                if (r + d[0], c + d[1]) in open_cells)
    return edges - len(open_cells) + 1

print('starter loops:', count_loops(STARTER), '  my maze loops:', count_loops(MY_MAZE))
for row in MY_MAZE:
    print(row)

Now turn your maze into a problem and run both searches on it. Fill in the **two** blanks.

In [ ]:
MY_WALLS, MY_START, MY_GOAL = parse(MY_MAZE)     # blank 1: read the maze
my_problem = MazeProblem(MY_WALLS, MY_START, MY_GOAL)   # blank 2: build the problem

my_results = {}
for strat in ['bfs', 'dfs']:
    path, expanded = search(my_problem, strat)
    my_results[strat] = (len(path), expanded)
    print('%-4s  path length %2d   nodes expanded %3d' % (strat.upper(), len(path), expanded))
print()
for strat in ['bfs', 'dfs']:
    show(MY_MAZE, search(my_problem, strat)[0], '%s route on my maze' % strat.upper())

In [ ]:
# CHECK Task 2
def _task2():
    rows = len(MY_MAZE)
    same_width = len({len(row) for row in MY_MAZE}) == 1
    check('at least 15 rows', rows >= 15, True, 'add rows to MY_MAZE')
    check('every row the same length', same_width, True, 'count the characters in each string')
    if not same_width:
        return
    check('at least 15 columns', len(MY_MAZE[0]) >= 15, True, 'make the strings longer')
    border = (set(MY_MAZE[0]) == {'#'} and set(MY_MAZE[-1]) == {'#'}
              and all(row[0] == '#' and row[-1] == '#' for row in MY_MAZE))
    check('outer border is all walls', border, True, 'the first/last row and first/last column must be #')
    text = ''.join(MY_MAZE)
    check('exactly one S and one G', (text.count('S'), text.count('G')), (1, 1), 'one start, one goal')
    if len(MY_MAZE) == len(STARTER) and len(MY_MAZE[0]) == len(STARTER[0]):
        changed = sum(a != b for ra, rb in zip(MY_MAZE, STARTER) for a, b in zip(ra, rb))
    else:
        changed = 99      # a different size counts as changed
    check('changed at least 2 cells from STARTER', changed >= 2, True,
          'open (#->.) or close (.->#) at least two cells')
    check('the goal can be reached', search(my_problem, 'bfs')[0] is not None, True,
          'do not wall the goal off')
    check('at least 2 loops (several routes)', count_loops(MY_MAZE) >= 2, True,
          'open a wall between two corridors that run side by side')
    print('\nyour maze: %d x %d, %d cells changed, %d loops'
          % (len(MY_MAZE), len(MY_MAZE[0]), changed, count_loops(MY_MAZE)))
_task2()

Copy your four numbers (path length and nodes expanded, for BFS and DFS) into the
observations table in Section 7.

### Task 3 — Make depth-first look bad

> **Marks:** 5

You need a maze where DFS returns a route **at least twice as long** as BFS.

**Part A — which way does DFS go first?** In `actions`, the moves are tried in the order
**up, down, left, right**, and each is `append`ed to the frontier in that order. DFS takes
the **newest** path — the one appended **last**. So from any cell, which direction does DFS
try first?

In [ ]:
dfs_first_direction = 'right'      # 'up', 'down', 'left' or 'right'

In [ ]:
# CHECK Task 3, Part A
_order = ['up', 'down', 'left', 'right']
check('DFS tries first', dfs_first_direction, _order[-1],
      'the move appended LAST is popped FIRST by pop()')

**Part B — set the trap.** Below is a maze that is one long loop of corridor, with **no
goal yet**. DFS will head **right** from `S` and go all the way round. Choose where to put
`G` so that BFS finds it in a few steps but DFS only finds it after the long way round.

> **Hint:** BFS finds a goal that is close to `S` quickly. Put `G` close to `S`, but **not**
> in the direction DFS goes first.

In [ ]:
TRAP_NO_GOAL = [
    '#############',
    '#S..........#',
    '#.#########.#',
    '#.#########.#',
    '#.#########.#',
    '#...........#',
    '#############',
]

GOAL_SPOT = (3, 1)        # three steps below S: close to S, but NOT in the direction DFS goes first

grid = [list(row) for row in TRAP_NO_GOAL]
grid[GOAL_SPOT[0]][GOAL_SPOT[1]] = 'G'
TRAP = [''.join(row) for row in grid]
trap = MazeProblem(*parse(TRAP))

trap_bfs, _ = search(trap, 'bfs')
trap_dfs, _ = search(trap, 'dfs')
ratio = len(trap_dfs) / len(trap_bfs)
print('BFS path length', len(trap_bfs), '   DFS path length', len(trap_dfs), '   ratio %.1f' % ratio)
show(TRAP, trap_bfs, 'BFS route on the trap maze')
show(TRAP, trap_dfs, 'DFS route on the trap maze')


In [ ]:
# CHECK Task 3, Part B
check('goal placed on an open cell', TRAP_NO_GOAL[GOAL_SPOT[0]][GOAL_SPOT[1]], '.',
      'choose a (row, col) where TRAP_NO_GOAL has a dot')
check('DFS route at least twice as long as BFS', ratio >= 2, True,
      'put G a few steps DOWN from S - DFS goes right first and takes the long way round')

**Part C — explanation**

DFS returned a route 9 times as long as BFS (27 cells against 3). This happened because my maze has
2 routes to the goal: a short one going down from `S`, and a long one going right and all the way round the loop. DFS always tries right first and **commits** to it — it does not notice that a
route is long, it simply returns the first route it reaches. BFS explores in layers (all routes of 1 step,
then 2, …), so it reaches the goal by the short route first.
If the moves were tried in a different order, DFS would try down first and find the short route immediately, so the trap only works because of the move order (it would then be enough to move `G` to the other side).

### Task 4 — Remove the repeated-state check

> **Marks:** 4

Our search has **two** different guards (Lecture 12):

| Guard | Code | What it stops |
|---|---|---|
| **cycle check** | `if nxt not in path` | one path walking back over **its own** cells |
| **repeated-state check** | the `visited` set | **two different paths** to the same cell both being expanded |

With `use_visited=False` only the second one is switched off. We test it on four mazes,
including a new one: an open **room** with no inner walls, where there are very many
different routes to every cell.

**Part A — predict first.** In the open room, will BFS expand **more**, **fewer** or the
**same** number of nodes without the repeated-state check?

In [ ]:
prediction_room_bfs = 'more'     # 'more', 'fewer' or 'same'

**Part B — run it.** Fill in the **one** blank so the second count is made *without* the check.

In [ ]:
ROOM = [
    '#######',
    '#S....#',
    '#.....#',
    '#.....#',
    '#....G#',
    '#######',
]
room = MazeProblem(*parse(ROOM))

no_check = {}
print('%-6s %-5s %14s %17s' % ('maze', 'strat', 'with check', 'without check'))
for name, prob in [('given', problem), ('trap', trap), ('room', room), ('mine', my_problem)]:
    for strat in ['bfs', 'dfs']:
        _, with_check = search(prob, strat, use_visited=True)
        _, without = search(prob, strat, use_visited=False)
        no_check[(name, strat)] = (with_check, without)
        print('%-6s %-5s %14d %17d   (x%.1f)'
              % (name, strat.upper(), with_check, without, without / with_check))

In [ ]:
# CHECK Task 4
_w, _wo = no_check[('room', 'bfs')]
check('the second count really is without the check', _wo != _w, True,
      'blank: use_visited=False')
check('your prediction for the room', prediction_room_bfs,
      'more' if _wo > _w else ('fewer' if _wo < _w else 'same'),
      'look at the room / BFS row of the table')

**Part C — explanation**

In the open room, BFS expanded 20 nodes with the check and 209 without it. Without
the check, every **different route** to a cell puts a separate copy of that cell on the frontier,
and each copy is expanded separately. In the room there are very many different routes to every cell (a grid with no inner walls).

On the given maze and the trap maze the count did not change, because there is only one simple route to each cell (the corridors form a tree or one single loop), so no cell is reached by two different paths before the goal —
so the cycle check `nxt not in path` was already enough.

DFS changed less (or not at all) because it dives down one route and reaches the goal before it ever meets a second route to the same cell, so there is nothing for the check to skip.

In the words of Lecture 12: the repeated-state check turns a search over paths into a
search over states (cells), and there are far fewer states than paths.

### Task 5 — Put the path into the state

> **Marks:** 4

Now we make the mistake **on purpose**. The state becomes a pair:
`(position, path_so_far)`. Everything else stays the same. Will the `visited` check still
save any work?

**Part A — build the broken problem.** Fill in the **two** blanks.

In [ ]:
class PathInStateProblem:
    """The same maze, with the route folded into the state. A deliberate mistake."""
    def __init__(self, walls, start, goal):
        self.walls, self.goal = walls, goal
        self.start = (start, (start,))          # blank 1: (position, path so far) - a 1-item tuple

    def start_state(self):
        return self.start

    def is_goal(self, state):
        position, path_so_far = state
        return position == self.goal            # blank 2: compare only the POSITION

    def actions(self, state):
        (r, c), path_so_far = state
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nxt = (r + dr, c + dc)
            if nxt not in self.walls and nxt not in path_so_far:
                yield (nxt, path_so_far + (nxt,)), 1

broken = PathInStateProblem(MY_WALLS, MY_START, MY_GOAL)
print('a broken state looks like:', broken.start_state())

**Part B — measure it.** `search_counted` (given) is our `search`, plus one extra counter:
`hits` = how many times the `visited` check **actually skipped** a repeated state — that is,
how often it saved work.

In [ ]:
def search_counted(problem, strategy='bfs'):
    frontier = [[problem.start_state()]]
    visited, expanded, hits = set(), 0, 0
    while frontier:
        path = frontier.pop(0) if strategy == 'bfs' else frontier.pop()
        state = path[-1]
        if state in visited:
            hits += 1                     # the check saved work here
            continue
        visited.add(state)
        expanded += 1
        if problem.is_goal(state):
            return len(path), expanded, hits
        for nxt, cost in problem.actions(state):
            if nxt not in path:
                frontier.append(path + [nxt])
    return None, expanded, hits

t5 = {}
print('%-24s %12s %10s %14s' % ('state design (BFS)', 'path length', 'expanded', 'visited hits'))
for label, prob in [('(row, col)', my_problem), ('(row, col), path', broken)]:
    t5[label] = search_counted(prob, 'bfs')
    print('%-24s %12d %10d %14d' % ((label,) + t5[label]))

In [ ]:
# CHECK Task 5
check('broken state still finds a route of the same length',
      t5['(row, col), path'][0], t5['(row, col)'][0], 'blank 2: return position == self.goal')
check('the visited check NEVER fires with the path in the state',
      t5['(row, col), path'][2], 0, 'blank 1: (start, (start,)) - note the comma')
check('...but it does fire with the plain state', t5['(row, col)'][2] > 0, True,
      'run the Task 2 cells first')

**Part C — explanation**

With the plain state `(row, col)`, the visited check fired 4 times. With the path in
the state it fired 0 times, and the search expanded 109 nodes instead of 89.
The answer was still correct (the same shortest route, 33 cells).

The check stops working because two states are equal only if both the position and the whole path so far are equal. Two
different routes to the same cell have different histories, so the two states are never equal and the check never recognises a repeat …

This is worse than deleting the check, because the program still carries the visited set (extra memory and work) and looks as if it is protected, yet gives no protection at all, and every stored state also holds a whole path.

## 7. Observations

Results from running the notebook (path length counts cells in the path).

| Maze | Strategy | Nodes expanded | Path length | Shortest? |
|---|---|---|---|---|
| Given maze | BFS | 24 | 12 | Yes |
| Given maze | DFS | 26 | 16 | No |
| My maze    | BFS | 89 | 33 | Yes |
| My maze    | DFS | 56 | 53 | No |
| Trap maze  | BFS | 4 | 3 | Yes |
| Trap maze  | DFS | 27 | 27 | No |

**Without the repeated-state check (Task 4)**

| Maze | Strategy | Nodes expanded (with check) | Nodes expanded (without) |
|---|---|---|---|
| Given maze | BFS | 24 | 24 |
| Given maze | DFS | 26 | 26 |
| Open room  | BFS | 20 | 209 |
| Open room  | DFS | 16 | 16 |
| My maze    | BFS | 89 | 109 |
| My maze    | DFS | 56 | 56 |

**Path in the state (Task 5, BFS on my maze)**

| State design | Path length | Nodes expanded | Times the `visited` check fired |
|---|---|---|---|
| `(row, col)` | 33 | 89 | 4 |
| `(row, col), path` | 33 | 109 | 0 |

## 8. Discussion

1. The line is `path = frontier.pop(0)` for BFS and `path = frontier.pop()` for DFS. Everything else in the function is identical: the visited check, the goal test, the expansion and the cycle check do not change.

2. No, because BFS finds the route with the fewest steps, not the lowest total cost. If one step cost 10, a route with fewer steps could cost more than a longer route of cheap steps, and BFS would still return the fewer-step one. (Lowest-cost-first search, which always takes the cheapest path from the frontier, is needed then.)

3. In the open room it went from 20 to 209, because without the check every different route to a cell puts another copy of that cell on the frontier and each copy is expanded again. On mazes with essentially one route per cell (the given maze, the trap) nothing changed, and on my maze (a few loops) it rose only from 89 to 109.

## 9. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not
whether you can recite the code.

1. What are the four components of a search problem, for your maze?
2. Show me the one line that turns this into depth-first search.
3. Why is the path not part of the state?
4. Why do we count nodes expanded rather than measuring the running time?
5. Depth-first search is incomplete. Show me a maze where that matters.
6. What would you change to make this lowest-cost-first search?

## 10. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel (Kernel → Restart & Run All)
- [ ] Every CHECK cell prints `OK`
- [ ] Task 2: your maze is at least 15 × 15, differs from the starter, and has at least 2 loops
- [ ] The observations tables are filled in with your actual results
- [ ] The Part B/C explanations and the discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P04a_<your-roll-number>.ipynb`

In [2]:
NAME = "Roha"
ROLL_NUMBER = "2025-cse-024"
print(NAME, ROLL_NUMBER)

Roha 2025-cse-024
